# 03 - Silver transform

Cleans bronze rows and MERGEs them into `silver.jobs_clean` (one row per `job_key`).

**What gets processed:** bronze rows whose `_job_run_id` has not yet been handled by a successful silver run (tracked in `ops.pipeline_runs.details.bronze_run_ids`). A failed run is automatically picked up next time.

**Cleaning**
| Step | Rule |
|---|---|
| Trim | All strings trimmed, empty -> `NULL`, whitespace in descriptions collapsed |
| Reject | Missing `job_id`, `title`, `ats_type` or `job_url` -> `silver.jobs_rejected` with reason |
| Company | `LinkedIn \| ` prefix removed, missing -> `Unknown` |
| LinkedIn ids | Legacy slug ids (`title-at-company-4012345678`) -> numeric id, tracking params removed from URL |
| Key | `job_key = sha2(lower(ats_type) \| lower(company_name) \| job_id)` |
| Dates | `dd/MM/yyyy` -> `DATE`, unparseable (e.g. `Posted 30+ Days Ago`) -> `NULL` |
| LinkedIn criteria | Trailing *Seniority level / Employment type / Job function / Industries* block parsed into columns and removed from the description |
| Location | `is_india` via `ops.ref_india_locations` (handles `Gurgaon`, `Bangalore`...), `country` = last part of the first location |
| Dedupe | Latest sighting per `job_key` wins; `first_seen_at`, `last_seen_at`, `times_seen` maintained across runs |

**Job task environment:** no extra dependencies needed.

In [ ]:
%run ./_common

In [ ]:
import re

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "Full refresh (truncate silver, reprocess all bronze)")
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

BRONZE_TABLE = table("bronze", "jobs_raw")
SILVER_TABLE = table("silver", "jobs_clean")
REJECTED_TABLE = table("silver", "jobs_rejected")
RUNS_TABLE = table("ops", "pipeline_runs")
INDIA_REF_TABLE = table("ops", "ref_india_locations")

SILVER_COLUMNS = [
    "job_key", "source", "ats_type", "company_name", "company_url", "job_id", "title", "location", "country", "is_india",
    "posted_date", "posted_days", "collected_on", "job_url", "description", "description_raw_length", "description_hash",
    "seniority_level", "employment_type", "job_function", "industries", "dq_missing_description",
    "first_seen_at", "last_seen_at", "times_seen", "_source_file", "_row_hash", "_ingested_at", "_job_run_id", "_updated_at",
]

print(f"source={BRONZE_TABLE} target={SILVER_TABLE} full_refresh={FULL_REFRESH}")

## Find unprocessed bronze runs

In [ ]:
if FULL_REFRESH:
    spark.sql(f"TRUNCATE TABLE {SILVER_TABLE}")
    spark.sql(f"TRUNCATE TABLE {REJECTED_TABLE}")
    print("Silver tables truncated; all bronze runs will be reprocessed")
    processed_runs = spark.createDataFrame([], "run_id STRING")
else:
    processed_runs = spark.sql(f"""
        SELECT explode(from_json(details, 'struct<bronze_run_ids:array<string>>').bronze_run_ids) AS run_id
        FROM {RUNS_TABLE}
        WHERE task = 'silver_transform' AND status = 'SUCCESS'
    """)

pending_run_ids = sorted(
    row["_job_run_id"]
    for row in spark.table(BRONZE_TABLE).select("_job_run_id").distinct()
    .join(processed_runs, F.col("_job_run_id") == F.col("run_id"), "left_anti")
    .collect()
)
print(f"Pending bronze runs: {pending_run_ids or 'none'}")

## Cleaning helpers

In [ ]:
CRITERIA_LABELS = ["Seniority level", "Employment type", "Job function", "Industries"]
LABEL_PATTERN = "|".join(re.escape(label) for label in CRITERIA_LABELS)
TRAILING_LABELS_RE = re.compile(rf"((?:{LABEL_PATTERN})(?: (?:{LABEL_PATTERN}))*)\s*$")

criteria_schema = StructType([
    StructField("description", StringType()),
    StructField("seniority_level", StringType()),
    StructField("employment_type", StringType()),
    StructField("job_function", StringType()),
    StructField("industries", StringType()),
])


@F.udf(returnType=criteria_schema)
def split_linkedin_criteria(text):
    """LinkedIn descriptions end with 'L1 V1 L2 V2 ... V1 V2 ... L1 L2 ...'. Extract V1..Vn and drop the block."""
    if not text:
        return (text, None, None, None, None)
    trailing = TRAILING_LABELS_RE.search(text)
    if not trailing:
        return (text, None, None, None, None)

    labels = re.findall(LABEL_PATTERN, trailing.group(1))
    pattern = (
        " (.+?) ".join(re.escape(label) for label in labels)
        + " (.+?) "
        + " ".join(rf"\{index + 1}" for index in range(len(labels)))
        + r"\s+" + re.escape(trailing.group(1)) + r"\s*$"
    )
    block = re.search(pattern, text)
    if not block:
        return (text[:trailing.start()].strip() or None, None, None, None, None)

    values = dict(zip(labels, (value.strip() for value in block.groups())))
    return (
        text[:block.start()].strip() or None,
        values.get("Seniority level"),
        values.get("Employment type"),
        values.get("Job function"),
        values.get("Industries"),
    )


def clean_string(column):
    trimmed = F.trim(F.col(column))
    return F.when(trimmed == "", None).otherwise(trimmed)


def parse_dmy(column):
    return F.expr(f"CAST(try_to_timestamp({column}, 'dd/MM/yyyy') AS DATE)")

## Transform and MERGE

In [ ]:
with task_run("silver_transform") as metrics:
    metrics["details"]["bronze_run_ids"] = pending_run_ids

    if not pending_run_ids:
        metrics.update(rows_in=0, rows_out=0, rows_rejected=0, message="nothing to process")
    else:
        bronze = spark.table(BRONZE_TABLE).where(F.col("_job_run_id").isin(pending_run_ids))

        base = bronze.select(
            *[clean_string(column).alias(column) for column in [
                "company_name", "company_url", "job_id", "title", "location", "job_url", "posted_date", "collected_on", "posted_days",
            ]],
            F.lower(clean_string("ats_type")).alias("ats_type"),
            F.regexp_replace(F.col("description"), r"\s+", " ").alias("description_raw"),
            "_rescued_data", "_source_file", "_source_system", "_file_modification_time", "_row_hash", "_ingested_at",
        )

        reasons = F.concat_ws(",", F.array(
            F.when(F.col("job_id").isNull(), F.lit("missing_job_id")),
            F.when(F.col("title").isNull(), F.lit("missing_title")),
            F.when(F.col("ats_type").isNull(), F.lit("missing_ats_type")),
            F.when(F.col("job_url").isNull(), F.lit("missing_job_url")),
        ))
        base = base.withColumn("reject_reason", F.when(reasons != "", reasons))

        rejected = base.where(F.col("reject_reason").isNotNull()).select(
            "reject_reason", "company_name", "ats_type", "job_id", "title", "job_url", "location",
            "_rescued_data", "_source_file", "_row_hash", "_ingested_at",
            F.lit(JOB_RUN_ID).alias("_job_run_id"), F.current_timestamp().alias("_rejected_at"),
        )
        rejected.write.mode("append").saveAsTable(REJECTED_TABLE)

        is_linkedin = F.col("ats_type") == "linkedin"
        company = F.trim(F.regexp_replace(F.col("company_name"), r"^LinkedIn(\s*\|\s*|$)", ""))

        valid = (
            base.where(F.col("reject_reason").isNull())
            .withColumn("company_name", F.when((company == "") | company.isNull(), F.lit("Unknown")).otherwise(company))
            .withColumn("job_id", F.when(is_linkedin & F.col("job_id").rlike("[0-9]+$"), F.regexp_extract("job_id", r"([0-9]+)$", 1)).otherwise(F.col("job_id")))
            .withColumn("job_url", F.when(is_linkedin, F.split("job_url", r"\?").getItem(0)).otherwise(F.col("job_url")))
            .withColumn("job_key", F.sha2(F.concat_ws("|", F.col("ats_type"), F.lower(F.col("company_name")), F.col("job_id")), 256))
            .withColumn("criteria", split_linkedin_criteria(F.col("description_raw")))
            .withColumn("description", F.col("criteria.description"))
            .withColumn("_seen_at", F.coalesce(F.col("_file_modification_time"), F.col("_ingested_at")))
        )

        # Location -> is_india using the reference aliases (word-boundary match)
        india_aliases = F.broadcast(spark.table(INDIA_REF_TABLE).select(F.lower("alias").alias("alias")).distinct())
        india_locations = (
            valid.select("location").where(F.col("location").isNotNull()).distinct()
            .crossJoin(india_aliases)
            .where(F.expr("lower(location) RLIKE concat('(^|[^a-z])', alias, '([^a-z]|$)')"))
            .select("location").distinct().withColumn("is_india", F.lit(True))
        )
        first_location = F.trim(F.split(F.col("location"), ";").getItem(0))
        last_part = F.trim(F.element_at(F.split(first_location, ","), -1))

        valid = (
            valid.join(india_locations, "location", "left")
            .withColumn("is_india", F.coalesce(F.col("is_india"), F.lit(False)))
            .withColumn("country", F.when(F.col("is_india"), F.lit("India"))
                        .when(F.col("location").isNull() | F.col("location").rlike(r"(?i)^[0-9]+ locations?$"), F.lit(None))
                        .otherwise(last_part))
        )

        # Dedupe within this batch: latest sighting wins, keep sighting stats
        latest_first = Window.partitionBy("job_key").orderBy(F.col("_seen_at").desc(), F.col("_ingested_at").desc(), F.col("_row_hash"))
        sightings = valid.groupBy("job_key").agg(
            F.min("_seen_at").alias("first_seen_at"),
            F.max("_seen_at").alias("last_seen_at"),
            F.countDistinct("_source_file").cast("int").alias("times_seen"),
        )
        updates = (
            valid.withColumn("_rank", F.row_number().over(latest_first)).where("_rank = 1")
            .join(sightings, "job_key")
            .select(
                "job_key",
                F.coalesce(F.col("ats_type"), F.col("_source_system")).alias("source"),
                "ats_type", "company_name", "company_url", "job_id", "title", "location", "country", "is_india",
                parse_dmy("posted_date").alias("posted_date"),
                F.expr("CAST(TRY_CAST(posted_days AS DOUBLE) AS INT)").alias("posted_days"),
                parse_dmy("collected_on").alias("collected_on"),
                "job_url", "description",
                F.length("description_raw").alias("description_raw_length"),
                F.sha2(F.coalesce(F.col("description"), F.lit("")), 256).alias("description_hash"),
                F.col("criteria.seniority_level").alias("seniority_level"),
                F.col("criteria.employment_type").alias("employment_type"),
                F.col("criteria.job_function").alias("job_function"),
                F.col("criteria.industries").alias("industries"),
                (F.col("description").isNull() | (F.length("description") < 50)).alias("dq_missing_description"),
                "first_seen_at", "last_seen_at", "times_seen", "_source_file", "_row_hash", "_ingested_at",
                F.lit(JOB_RUN_ID).alias("_job_run_id"),
                F.current_timestamp().alias("_updated_at"),
            )
        )
        updates.createOrReplaceTempView("silver_updates")

        refresh_columns = [c for c in SILVER_COLUMNS if c not in ("job_key", "first_seen_at", "last_seen_at", "times_seen")]
        full_update = ", ".join(f"t.{c} = s.{c}" for c in refresh_columns)
        merge_result = spark.sql(f"""
            MERGE INTO {SILVER_TABLE} t
            USING silver_updates s
            ON t.job_key = s.job_key
            WHEN MATCHED AND s.last_seen_at >= t.last_seen_at THEN UPDATE SET
              {full_update},
              t.first_seen_at = least(t.first_seen_at, s.first_seen_at),
              t.last_seen_at = s.last_seen_at,
              t.times_seen = t.times_seen + s.times_seen
            WHEN MATCHED THEN UPDATE SET
              t.first_seen_at = least(t.first_seen_at, s.first_seen_at),
              t.times_seen = t.times_seen + s.times_seen,
              t._job_run_id = s._job_run_id,
              t._updated_at = s._updated_at
            WHEN NOT MATCHED THEN INSERT ({", ".join(SILVER_COLUMNS)})
              VALUES ({", ".join(f"s.{c}" for c in SILVER_COLUMNS)})
        """).first().asDict()

        rows_in = bronze.count()
        rows_rejected = spark.table(REJECTED_TABLE).where(F.col("_job_run_id") == JOB_RUN_ID).count()
        metrics.update(
            rows_in=rows_in,
            rows_out=int(merge_result.get("num_affected_rows", 0)),
            rows_rejected=rows_rejected,
        )
        metrics["details"].update(
            unique_jobs_in_batch=updates.count(),
            inserted=int(merge_result.get("num_inserted_rows", 0)),
            updated=int(merge_result.get("num_updated_rows", 0)),
        )

In [ ]:
stop_if_job()  # scheduled runs end here; the cells below are for manual inspection

## Inspect

In [ ]:
silver = spark.table(SILVER_TABLE)
display(
    silver.groupBy("source")
    .agg(
        F.count("*").alias("jobs"),
        F.sum(F.col("is_india").cast("int")).alias("india_jobs"),
        F.sum(F.col("dq_missing_description").cast("int")).alias("missing_description"),
        F.sum(F.col("posted_date").isNull().cast("int")).alias("missing_posted_date"),
        F.sum(F.col("seniority_level").isNotNull().cast("int")).alias("with_linkedin_criteria"),
        F.round(F.avg("times_seen"), 2).alias("avg_times_seen"),
    )
    .orderBy("source")
)

In [ ]:
display(
    silver.where("source = 'linkedin'")
    .select("company_name", "job_id", "title", "seniority_level", "employment_type", "job_function", "industries",
            F.substring("description", -120, 120).alias("description_tail"))
    .limit(10)
)

In [ ]:
display(silver.groupBy("source", "country").count().orderBy(F.col("count").desc()).limit(25))

In [ ]:
display(spark.table(REJECTED_TABLE).groupBy("reject_reason", "ats_type").count().orderBy(F.col("count").desc()))

In [ ]:
display(spark.table(RUNS_TABLE).orderBy(F.col("started_at").desc()).limit(10))